In [2]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm catboost xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.8 MB/s eta 0:00:00


In [3]:
import sys
!{sys.executable} -m pip install -q imblearn

In [4]:
from imblearn.over_sampling import SMOTE

In [5]:
import os
import re
import time
import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, List
from collections import defaultdict
from itertools import combinations

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    r2_score,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE


# =============================================================================
# CONFIG
# =============================================================================
#
# OBJECTIF DE CE NOTEBOOK (v3 - folds équilibrés, sans Google Trends) :
#   1. Charger les données Yahoo/FRED uniquement (Google Trends RETIRÉ : la
#      source s'est révélée bloquée par rate-limit Google et n'a apporté
#      aucune feature exploitable lors du run précédent).
#   2. Pour chaque régime (CALM/NORMAL/STRESS/GLOBAL) et chaque horizon
#      (1, 3, 5 jours), sweep complet des 11 fenêtres de train (2000-2010),
#      score composite Spearman+MutualInfo+Stabilité, avec génération
#      d'interactions (ratio/différence) sur le top 30 - inchangé.
#   3. RECOMMANDATION (cette version) : walk-forward à FOLDS ADAPTATIFS.
#      Découverte du run précédent : un découpage calendaire fixe (6 mois)
#      laisse certains folds sans aucune observation CALM ou STRESS (ex:
#      fold "jan-juin 2025" entièrement dominé par un épisode STRESS de 54
#      jours -> 0 jour CALM testable ; "jan-juin 2024" sans aucun jour
#      STRESS -> 0 jour STRESS testable). Les folds sont maintenant calculés
#      pour garantir un minimum d'observations CALM ET STRESS par fold
#      (MIN_OBS_PER_REGIME_PER_FOLD), au prix de folds de durée inégale.
#   4. Sortie : un seul fichier Excel récapitulatif + suggestions automatiques.

OUTPUT_DIR = Path("/content/outputs_v20_balanced_folds_no_trends")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START = "2000-01-01"   # utilisé uniquement pour le téléchargement initial
TEST_DATE = "2024-01-01"     # fixe pour toutes les phases

# --- Walk-forward à FOLDS ADAPTATIFS (remplace les folds calendaires fixes) ---
# 4 folds expanding (le train s'étend toujours), mais les bornes de TEST sont
# déterminées dynamiquement pour garantir au moins MIN_OBS_PER_REGIME_PER_FOLD
# jours CALM et STRESS dans chaque fold de test. Calculées une fois sur la
# grille de dates commune aux 3 horizons (voir build_balanced_walkforward_folds).
N_WALKFORWARD_FOLDS = 4
MIN_OBS_PER_REGIME_PER_FOLD = 15  # minimum de jours CALM et STRESS par fold de test
WALKFORWARD_TEST_START = "2024-01-01"  # début de la zone de test globale (inchangé)

# --- Phase 1 : fenêtres candidates pour la sélection de la "meilleure fenêtre"
# par régime (sweep complet, comme avant).
TRAIN_START_CANDIDATES = [f"{year}-01-01" for year in range(2000, 2011)]  # 2000..2010

FEATURE_PREFILTER_TOP_N = 300

YF_CHUNK_SIZE = 40
SLEEP_BETWEEN_CHUNKS = 1.0

MIN_COLUMN_COVERAGE = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365

VIX_FLAT_PCT_THRESHOLD = 0.0

# --- Sélection de features : plage testée et nombre final retenu ---
MIN_N_FEATURES_TO_TEST = 5
MAX_FEATURES_TO_TEST = 30   # top 30

# --- Score composite de qualité des features (Phase 1) ---
FEATURE_SCORE_WEIGHTS = {
    "spearman": 0.5,
    "mutual_info": 0.3,
    "stability": 0.2,
}
N_STABILITY_SUBPERIODS = 3

# Paramètre obligatoire de TargetBuilder.__init__ même en mode "fixed" (la
# signature l'exige, simplement inutilisé dans la branche fixed).
ROLLING_QUANTILE_WINDOW = 504

np.random.seed(RANDOM_STATE)

FRED_API_KEY = os.getenv("FRED_API_KEY")
if FRED_API_KEY:
    os.environ["FRED_API_KEY"] = FRED_API_KEY


In [6]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [7]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [8]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [9]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [10]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [11]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [12]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [13]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [14]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [15]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [16]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [17]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [18]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [19]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40
[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LVRK"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CBOT_W"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40
[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [20]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [21]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6734, 1180), features: 985


In [22]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [23]:
# =============================================================================
# 5 algos (LogisticRegression réintégrée) testés pour chaque régime.
# SMOTE appliqué SYSTÉMATIQUEMENT et sans condition sur les 4 régimes
# (CALM/NORMAL/STRESS/GLOBAL) - voir la boucle d'entraînement en Phase 2.
# =============================================================================

ALL_REGIMES = ["CALM", "NORMAL", "STRESS", "GLOBAL"]
ALL_ALGOS = ["XGBoost", "LightGBM", "GradientBoosting", "RandomForest", "LogisticRegression"]

MAX_FEATURES_TO_TEST = 20  # phase 2 : on teste N=1..20 sur la fenêtre gagnante


In [24]:
# =============================================================================
# PHASE 1 (renforcee, multi-horizon) : pour CHAQUE horizon de prediction
# (1, 3, 5 jours), pour chaque (train_start, regime), on calcule un SCORE
# COMPOSITE de qualite des features en DEUX temps :
#
#   Etape A : select_and_filter_features sur les features de BASE (top 30,
#             comme avant) -> liste ordonnee "de base".
#   Etape B (RECOMMANDATION 2) : generation de toutes les paires (ratio +
#             difference) sur ces features de base, puis re-passage dans
#             select_and_filter_features sur (base + interactions) ->
#             liste finale, qui peut donc contenir des interactions si
#             elles dominent certaines features brutes.
#
# Score composite = Spearman (50%) + Mutual Information (30%) + Stabilite
# temporelle sur N_STABILITY_SUBPERIODS sous-periodes (20%), chaque
# composante normalisee min-max avant ponderation.
#
# Le sweep Phase 1 tourne ENTIEREMENT pour chaque horizon (pas de reemploi
# entre horizons) - feature set optimal propre a chaque horizon.
# =============================================================================

def compute_stability_score(df_train_regime: pd.DataFrame, features: list,
                             target_col: str = "VIX_Direction",
                             n_subperiods: int = 3) -> pd.Series:
    """Retourne, pour chaque feature, un score de stabilite temporelle de sa
    correlation Spearman au target (1 = parfaitement stable, 0 = tres instable)."""
    df_sorted = df_train_regime.sort_index()
    n = len(df_sorted)
    if n < n_subperiods * 10:
        return pd.Series(1.0, index=features)

    boundaries = np.linspace(0, n, n_subperiods + 1).astype(int)
    subperiod_corrs = []

    for i in range(n_subperiods):
        sub_df = df_sorted.iloc[boundaries[i]:boundaries[i + 1]]
        if len(sub_df) < 10:
            continue
        cols = [f for f in features if f in sub_df.columns] + [target_col]
        sub_clean = sub_df[cols].dropna()
        if len(sub_clean) < 10 or sub_clean[target_col].nunique() < 2:
            continue
        corr = sub_clean.corr(method="spearman")[target_col].drop(target_col, errors="ignore")
        subperiod_corrs.append(corr)

    if len(subperiod_corrs) < 2:
        return pd.Series(1.0, index=features)

    corr_matrix = pd.concat(subperiod_corrs, axis=1)
    corr_std = corr_matrix.std(axis=1, skipna=True).fillna(1.0)
    stability_score = 1.0 / (1.0 + corr_std)
    return stability_score.reindex(features).fillna(0.5)


def minmax_normalize(series: pd.Series) -> pd.Series:
    rng = series.max() - series.min()
    if rng == 0 or pd.isna(rng):
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / rng


def compute_composite_score(df_train_regime: pd.DataFrame, ranked_features: list) -> dict:
    """Calcule le score composite (Spearman+MI+Stabilite) pour une liste de
    features deja selectionnee. Retourne un dict avec le score et ses 3 composantes."""
    df_corr = df_train_regime[ranked_features + ["VIX_Direction"]].dropna()
    spearman_scores = df_corr.corr(method="spearman")["VIX_Direction"].abs()
    spearman_scores = spearman_scores.drop("VIX_Direction", errors="ignore").reindex(ranked_features)

    df_mi = df_train_regime[ranked_features + ["VIX_Direction"]].dropna()
    try:
        mi_values = mutual_info_classif(
            df_mi[ranked_features].values, df_mi["VIX_Direction"].values,
            random_state=RANDOM_STATE, n_neighbors=3
        )
        mi_scores = pd.Series(mi_values, index=ranked_features)
    except Exception as e:
        print(f"[WARN] Mutual info echec: {e}. Score neutre applique.")
        mi_scores = pd.Series(0.5, index=ranked_features)

    stability_scores = compute_stability_score(
        df_train_regime, ranked_features, target_col="VIX_Direction",
        n_subperiods=N_STABILITY_SUBPERIODS
    )

    spearman_norm = minmax_normalize(spearman_scores)
    mi_norm = minmax_normalize(mi_scores)
    stability_norm = minmax_normalize(stability_scores)

    composite_per_feature = (
        FEATURE_SCORE_WEIGHTS["spearman"] * spearman_norm +
        FEATURE_SCORE_WEIGHTS["mutual_info"] * mi_norm +
        FEATURE_SCORE_WEIGHTS["stability"] * stability_norm
    )

    return {
        "quality_score": composite_per_feature.mean(),
        "spearman_mean": spearman_scores.mean(),
        "mi_mean": mi_scores.mean(),
        "stability_mean": stability_scores.mean(),
    }


def run_phase1_for_horizon(horizon_days: int, df_post_features: pd.DataFrame,
                            features_post_engineering: list):
    """Execute la Phase 1 complete (sweep Train_Start x regime, score
    composite avec interactions) pour UN horizon de prediction donne.
    Retourne (feature_quality_df, ranked_features_cache, winning_window_by_regime)."""
    print(f"\n{'#'*80}\n# PHASE 1 - HORIZON = {horizon_days} jour(s)\n{'#'*80}")

    feature_quality_rows = []
    ranked_features_cache = {}

    for train_start in TRAIN_START_CANDIDATES:
        df_window = df_post_features.loc[df_post_features.index >= pd.Timestamp(train_start)].copy()

        for regime in ALL_REGIMES:
            target_builder = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                                            rolling_window=ROLLING_QUANTILE_WINDOW,
                                            horizon_days=horizon_days)
            df = target_builder.build(df_window, train_end=TEST_DATE)

            train_mask = df.index < pd.Timestamp(TEST_DATE)
            df_train = df.loc[train_mask].copy()

            if regime == "GLOBAL":
                df_train_regime = df_train
            else:
                df_train_regime = df_train.loc[df_train["VIX_Regime"] == regime].copy()

            feats_all = [f for f in features_post_engineering if f in df_train_regime.columns]

            if len(df_train_regime) < 30 or "VIX_Direction" not in df_train_regime.columns:
                print(f"[WARN] h={horizon_days} train_start={train_start} regime={regime}: "
                      f"echantillon trop petit. Skip.")
                continue

            # --- Etape A : selection sur features de base ---
            base_ranked = select_and_filter_features(
                df=df_train_regime, all_features=feats_all, target_col="VIX_Direction",
                correlation_threshold_target=correlation_threshold_target,
                correlation_threshold_features=correlation_threshold_features,
                max_features_to_select=MAX_FEATURES_TO_TEST,
            )

            if not base_ranked:
                print(f"[WARN] h={horizon_days} train_start={train_start} regime={regime}: "
                      f"aucune feature de base selectionnee. Skip.")
                continue

            # --- Etape B (Recommandation 2) : interactions sur le top base_ranked ---
            interactions_df = generate_pairwise_interactions(df_train_regime, base_ranked)
            df_train_with_interactions = pd.concat([df_train_regime, interactions_df], axis=1)
            df_train_with_interactions = df_train_with_interactions.loc[:, ~df_train_with_interactions.columns.duplicated()]

            extended_features = base_ranked + list(interactions_df.columns)

            ranked_features = select_and_filter_features(
                df=df_train_with_interactions, all_features=extended_features, target_col="VIX_Direction",
                correlation_threshold_target=correlation_threshold_target,
                correlation_threshold_features=correlation_threshold_features,
                max_features_to_select=MAX_FEATURES_TO_TEST,
            )

            if not ranked_features:
                ranked_features = base_ranked  # repli si l'etape B echoue completement
                df_train_regime_final = df_train_regime
            else:
                df_train_regime_final = df_train_with_interactions

            n_interactions_kept = sum(1 for f in ranked_features if ("_div_" in f or "_minus_" in f))

            scores = compute_composite_score(df_train_regime_final, ranked_features)

            ranked_features_cache[(train_start, regime)] = {
                "features": ranked_features,
                "df_train_with_interactions": df_train_regime_final,  # garde pour reutilisation Phase 2
            }

            feature_quality_rows.append({
                "Horizon_Days": horizon_days,
                "Train_Start": train_start,
                "VIX_Regime": regime,
                "N_Features_Selected": len(ranked_features),
                "N_Interactions_Kept": n_interactions_kept,
                "Score_Spearman_Mean": scores["spearman_mean"],
                "Score_MutualInfo_Mean": scores["mi_mean"],
                "Score_Stability_Mean": scores["stability_mean"],
                "Quality_Score_Composite": scores["quality_score"],
                "Train_N": len(df_train_regime),
            })

            print(f"[SCORE] h={horizon_days} train_start={train_start} regime={regime}: "
                  f"{len(ranked_features)} features ({n_interactions_kept} interactions), "
                  f"composite={scores['quality_score']:.4f}, n={len(df_train_regime)}")

    feature_quality_df_h = pd.DataFrame(feature_quality_rows)

    winning_window_by_regime_h = {}
    for regime in ALL_REGIMES:
        sub = feature_quality_df_h[feature_quality_df_h["VIX_Regime"] == regime]
        if sub.empty:
            continue
        best_row = sub.loc[sub["Quality_Score_Composite"].idxmax()]
        winning_window_by_regime_h[regime] = best_row["Train_Start"]
        print(f"[WINNER h={horizon_days}] {regime}: Train_Start={best_row['Train_Start']} "
              f"(score={best_row['Quality_Score_Composite']:.4f})")

    return feature_quality_df_h, ranked_features_cache, winning_window_by_regime_h


# --- Exécution Phase 1 pour les 3 horizons ---
HORIZONS_TO_TEST = [1, 3, 5]

phase1_results_by_horizon = {}
for h in HORIZONS_TO_TEST:
    fq_df, rf_cache, win_windows = run_phase1_for_horizon(h, df_post_features, features_post_engineering)
    phase1_results_by_horizon[h] = {
        "feature_quality_df": fq_df,
        "ranked_features_cache": rf_cache,
        "winning_window_by_regime": win_windows,
    }

feature_quality_df = pd.concat(
    [v["feature_quality_df"] for v in phase1_results_by_horizon.values()], ignore_index=True
)
feature_quality_df.to_csv(OUTPUT_DIR / "phase1_feature_quality_by_window_and_horizon.csv", index=False)
print(f"\n[SAVE] phase1_feature_quality_by_window_and_horizon.csv ({len(feature_quality_df)} lignes)")



################################################################################
# PHASE 1 - HORIZON = 1 jour(s)
################################################################################
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.04, NORMAL [15.04-21.34), STRESS >= 21.34  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6733
[TARGET] Remaining non-flat rows: 6477
VIX_Regime
NORMAL    2349
CALM      2089
STRESS    2039
Name: count, dtype: int64
[INTERACTIONS] Génération de paires pour 20 features de base (190 paires x 2 opérations = 380 colonnes max)...
[INTERACTIONS] 380 colonnes d'interactions générées.
[SCORE] h=1 train_start=2000-01-01 regime=CALM: 20 features (20 interactions), composite=0.2893, n=1919
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.04, NORMAL [15.04-21.34), STRESS >= 21.34  (fixe, calculé sur train)
[TARGET] VIX flat 

In [25]:
# =============================================================================
# WALK-FORWARD A FOLDS ADAPTATIFS (remplace les folds calendaires fixes).
#
# Découverte du run précédent (folds calendaires 6 mois) : certains folds se
# retrouvaient avec 0 observation CALM ou STRESS (un épisode STRESS continu
# de 54 jours concentré sur un semestre épuise tous les jours non-STRESS du
# fold ; à l'inverse un semestre sans aucun épisode de stress laisse 0 jour
# STRESS testable). Conséquence : AUC_std très élevé voire NaN, métriques de
# Best_Model_Per_Regime non fiables pour CALM et STRESS.
#
# Algorithme : à partir de WALKFORWARD_TEST_START, on étend la fenêtre de
# test jour par jour jusqu'à atteindre AU MOINS MIN_OBS_PER_REGIME_PER_FOLD
# jours CALM et MIN_OBS_PER_REGIME_PER_FOLD jours STRESS dans cette fenêtre.
# Le fold suivant repart immédiatement après (test contigu, train expanding -
# train_end du fold N = test_end du fold N-1, comme le walk-forward calendaire
# avant). On répète pour N_WALKFORWARD_FOLDS folds.
#
# Le régime (CALM/NORMAL/STRESS) ne dépend PAS de l'horizon de prédiction
# (horizon_days n'affecte que VIX_Direction, pas VIX_Regime - voir
# TargetBuilder) : les folds sont donc calculés UNE SEULE FOIS et réutilisés
# pour les 3 horizons, garantissant une comparaison sur les mêmes périodes.
# =============================================================================

def build_balanced_walkforward_folds(df_post_features: pd.DataFrame,
                                      n_folds: int = 4,
                                      min_obs_per_regime: int = 15,
                                      test_start: str = "2024-01-01") -> list:
    """Retourne une liste de dicts {train_end, test_start, test_end}, comme
    WALKFORWARD_FOLDS avant, mais avec des bornes de test calculées pour
    garantir min_obs_per_regime jours CALM et STRESS par fold."""

    # Régime calculé sur toute la période disponible (mode fixed, seuils
    # calculés sur tout l'historique < test_start - cohérent avec le calcul
    # de régime utilisé partout ailleurs dans Phase 1/2).
    target_builder_for_folds = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                                              rolling_window=ROLLING_QUANTILE_WINDOW,
                                              horizon_days=1)  # horizon=1 ici : seul VIX_Regime nous intéresse, indépendant de l'horizon
    df_regime = target_builder_for_folds.build(df_post_features, train_end=test_start)

    regime_series = df_regime["VIX_Regime"].sort_index()
    test_zone = regime_series.loc[regime_series.index >= pd.Timestamp(test_start)]

    if test_zone.empty:
        raise ValueError(f"Aucune donnée disponible après test_start={test_start} pour construire les folds.")

    all_test_dates = test_zone.index.sort_values()

    folds = []
    current_train_end = pd.Timestamp(test_start)
    cursor_idx = 0  # position dans all_test_dates

    for fold_num in range(1, n_folds + 1):
        if cursor_idx >= len(all_test_dates):
            print(f"[WARN] Plus de données disponibles pour construire le fold {fold_num}/{n_folds}. "
                  f"Seulement {len(folds)} folds créés.")
            break

        fold_start_idx = cursor_idx
        calm_count = 0
        stress_count = 0
        end_idx = fold_start_idx

        # Étend la fenêtre jusqu'à atteindre le minimum requis pour les 2 régimes,
        # ou jusqu'à épuisement des données disponibles.
        while end_idx < len(all_test_dates):
            current_date = all_test_dates[end_idx]
            current_regime = regime_series.loc[current_date]
            if current_regime == "CALM":
                calm_count += 1
            elif current_regime == "STRESS":
                stress_count += 1
            end_idx += 1

            if calm_count >= min_obs_per_regime and stress_count >= min_obs_per_regime:
                break

        if calm_count < min_obs_per_regime or stress_count < min_obs_per_regime:
            print(f"[WARN] Fold {fold_num}: minimum non atteint avant épuisement des données "
                  f"(CALM={calm_count}, STRESS={stress_count}, requis={min_obs_per_regime}). "
                  f"Fold créé quand même avec ce qui est disponible (dernier fold partiel).")

        fold_test_start = all_test_dates[fold_start_idx]
        # test_end = jour APRES la dernière date incluse (borne exclusive, cohérent
        # avec le style [test_start, test_end) utilisé par WALKFORWARD_FOLDS avant)
        if end_idx < len(all_test_dates):
            fold_test_end = all_test_dates[end_idx]
        else:
            fold_test_end = all_test_dates[-1] + pd.Timedelta(days=1)

        folds.append({
            "train_end": current_train_end.strftime("%Y-%m-%d"),
            "test_start": fold_test_start.strftime("%Y-%m-%d"),
            "test_end": fold_test_end.strftime("%Y-%m-%d"),
            "n_calm": calm_count,
            "n_stress": stress_count,
            "n_total_days": end_idx - fold_start_idx,
        })

        print(f"[FOLD {fold_num}] {fold_test_start.date()} -> {fold_test_end.date()} "
              f"({end_idx - fold_start_idx} jours, CALM={calm_count}, STRESS={stress_count})")

        current_train_end = fold_test_end
        cursor_idx = end_idx

    return folds


WALKFORWARD_FOLDS = build_balanced_walkforward_folds(
    df_post_features,
    n_folds=N_WALKFORWARD_FOLDS,
    min_obs_per_regime=MIN_OBS_PER_REGIME_PER_FOLD,
    test_start=WALKFORWARD_TEST_START,
)

walkforward_folds_df = pd.DataFrame(WALKFORWARD_FOLDS)
walkforward_folds_df.to_csv(OUTPUT_DIR / "balanced_walkforward_folds.csv", index=False)
print(f"\n[SAVE] balanced_walkforward_folds.csv ({len(WALKFORWARD_FOLDS)} folds construits)")
display(walkforward_folds_df)


[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.04, NORMAL [15.04-21.34), STRESS >= 21.34  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6733
[TARGET] Remaining non-flat rows: 6477
VIX_Regime
NORMAL    2349
CALM      2089
STRESS    2039
Name: count, dtype: int64
[FOLD 1] 2024-01-02 -> 2025-03-05 (291 jours, CALM=144, STRESS=15)
[FOLD 2] 2025-03-05 -> 2025-12-22 (202 jours, CALM=15, STRESS=48)
[WARN] Fold 3: minimum non atteint avant épuisement des données (CALM=11, STRESS=28, requis=15). Fold créé quand même avec ce qui est disponible (dernier fold partiel).
[FOLD 3] 2025-12-22 -> 2026-06-30 (129 jours, CALM=11, STRESS=28)
[WARN] Plus de données disponibles pour construire le fold 4/4. Seulement 3 folds créés.

[SAVE] balanced_walkforward_folds.csv (3 folds construits)


,train_end,test_start,test_end,n_calm,n_stress,n_total_days
0,2024-01-01,2024-01-02,2025-03-05,144,15,291
1,2025-03-05,2025-03-05,2025-12-22,15,48,202
2,2025-12-22,2025-12-22,2026-06-30,11,28,129


In [26]:
# =============================================================================
# Récapitulatif consolidé : fenêtre gagnante par (horizon, régime).
# (Le calcul lui-même a déjà été fait dans run_phase1_for_horizon - ici on
# agrège juste les résultats des 3 horizons pour affichage/export.)
# =============================================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

for h in HORIZONS_TO_TEST:
    print(f"\n[PHASE 1 RESULTS] Score par fenêtre et régime, horizon={h}j :")
    sub_fq = phase1_results_by_horizon[h]["feature_quality_df"]
    pivot_quality = sub_fq.pivot_table(index="Train_Start", columns="VIX_Regime", values="Quality_Score_Composite")
    display(pivot_quality)

winner_rows = []
for h in HORIZONS_TO_TEST:
    for regime, ts in phase1_results_by_horizon[h]["winning_window_by_regime"].items():
        winner_rows.append({"Horizon_Days": h, "VIX_Regime": regime, "Winning_Train_Start": ts})

winner_summary_df = pd.DataFrame(winner_rows)
winner_summary_df.to_csv(OUTPUT_DIR / "phase1_winning_window_per_regime_and_horizon.csv", index=False)
print(f"\n[SAVE] phase1_winning_window_per_regime_and_horizon.csv")
display(winner_summary_df)



[PHASE 1 RESULTS] Score par fenêtre et régime, horizon=1j :


VIX_Regime,CALM,GLOBAL,NORMAL,STRESS
Train_Start,,,,
2000-01-01,0.289340,0.493624,0.516047,0.461982
2001-01-01,0.571418,0.496729,0.527803,0.372979
2002-01-01,0.348968,0.517858,0.438092,0.378198
2003-01-01,0.517534,0.519203,0.411615,0.405576
2004-01-01,0.532265,0.549875,0.465687,0.345379
2005-01-01,0.485474,0.514232,0.434283,0.353448
2006-01-01,0.417550,0.465942,0.379010,0.430240
2007-01-01,0.477518,0.517632,0.449883,0.396009
2008-01-01,0.422708,0.460297,0.445985,0.387069



[PHASE 1 RESULTS] Score par fenêtre et régime, horizon=3j :


VIX_Regime,CALM,GLOBAL,NORMAL,STRESS
Train_Start,,,,
2000-01-01,0.426943,0.508945,0.360412,0.449181
2001-01-01,0.448313,0.507443,0.381008,0.479334
2002-01-01,0.452809,0.518066,0.440339,0.363274
2003-01-01,0.426138,0.482848,0.403402,0.413191
2004-01-01,0.436747,0.478035,0.439092,0.399038
2005-01-01,0.420999,0.469772,0.322273,0.415722
2006-01-01,0.417394,0.429651,0.395652,0.407524
2007-01-01,0.463065,0.493165,0.404497,0.355725
2008-01-01,0.378592,0.495344,0.394989,0.442524



[PHASE 1 RESULTS] Score par fenêtre et régime, horizon=5j :


VIX_Regime,CALM,GLOBAL,NORMAL,STRESS
Train_Start,,,,
2000-01-01,0.426062,0.465576,0.429839,0.493223
2001-01-01,0.400506,0.477393,0.426554,0.412661
2002-01-01,0.451728,0.423329,0.519039,0.403327
2003-01-01,0.474561,0.480300,0.512890,0.401378
2004-01-01,0.420496,0.498749,0.463626,0.415613
2005-01-01,0.474054,0.492373,0.453477,0.325441
2006-01-01,0.504932,0.457297,0.468334,0.458813
2007-01-01,0.425565,0.445508,0.393568,0.590801
2008-01-01,0.374516,0.482938,0.381569,0.497444



[SAVE] phase1_winning_window_per_regime_and_horizon.csv


,Horizon_Days,VIX_Regime,Winning_Train_Start
0,1,CALM,2001-01-01
1,1,NORMAL,2001-01-01
2,1,STRESS,2009-01-01
3,1,GLOBAL,2010-01-01
4,3,CALM,2010-01-01
5,3,NORMAL,2010-01-01
6,3,STRESS,2010-01-01
7,3,GLOBAL,2002-01-01
8,5,CALM,2006-01-01
9,5,NORMAL,2002-01-01


In [27]:
# =============================================================================
# PHASE 2 (walk-forward, multi-horizon) : entraînement final.
#
# Pour CHAQUE horizon (1, 3, 5j), pour CHAQUE régime, sur SA fenêtre gagnante
# et SON feature set (base + interactions retenues, déjà calculés en Phase 1
# et mis en cache dans ranked_features_cache[h][(train_start, regime)]) :
#
#   Pour CHAQUE fold walk-forward (RECOMMANDATION 1, expanding window,
#   WALKFORWARD_FOLDS) :
#     - reconstruit le dataframe complet avec interactions sur la fenêtre
#       gagnante, split train (< fold.train_end) / test (fold.test_start,
#       fold.test_end)
#     - 5 algos x N=5..30 features, SMOTE systématique, GridSearchCV
#
# C'est le bloc le plus coûteux du notebook : horizons(3) x régimes(4) x
# folds(4) x algos(5) x N(26 valeurs) = jusqu'à 6240 fits GridSearchCV.
# =============================================================================

print("="*80)
print("PHASE 2 (walk-forward) : entraînement final")
print("="*80)

final_rows = []
model_number = 0

for h in HORIZONS_TO_TEST:
    winning_window_by_regime_h = phase1_results_by_horizon[h]["winning_window_by_regime"]
    ranked_features_cache_h = phase1_results_by_horizon[h]["ranked_features_cache"]

    for regime in ALL_REGIMES:
        train_start = winning_window_by_regime_h.get(regime)
        if train_start is None:
            print(f"[WARN] h={h} {regime}: pas de fenêtre gagnante désignée. Skip.")
            continue

        cache_entry = ranked_features_cache_h.get((train_start, regime))
        if cache_entry is None:
            print(f"[WARN] h={h} {regime}: pas d'entrée en cache pour {train_start}. Skip.")
            continue

        ranked_features = cache_entry["features"]
        # df avec interactions, calculé sur la fenêtre gagnante en Phase 1
        # (mais seulement la portion TRAIN de l'époque Phase 1 - on doit
        # reconstruire ici sur toute la période train+test pour le walk-forward).
        print(f"\n--- h={h}j {regime}: fenêtre gagnante Train_Start={train_start}, "
              f"{len(ranked_features)} features ---")

        df_window = df_post_features.loc[df_post_features.index >= pd.Timestamp(train_start)].copy()

        target_builder = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                                        rolling_window=ROLLING_QUANTILE_WINDOW,
                                        horizon_days=h)
        # train_end=TEST_DATE ici uniquement pour le calcul des seuils de régime
        # (cohérent avec Phase 1 - les seuils de régime ne doivent pas varier
        # par fold walk-forward, sinon la définition même du régime change en
        # cours de route, ce qui casserait la comparabilité des folds).
        df_full = target_builder.build(df_window, train_end=TEST_DATE)

        feats_all = [f for f in features_post_engineering if f in df_full.columns]
        df_full[feats_all] = df_full[feats_all].replace([np.inf, -np.inf], np.nan)

        # Régénère les interactions sur TOUT df_full (train+test), avec les
        # mêmes features de base que la Phase 1 - nécessaire pour avoir les
        # colonnes d'interaction disponibles sur la période test aussi.
        base_features_for_interactions = [f for f in ranked_features if "_div_" not in f and "_minus_" not in f]
        # Si ranked_features contient déjà des interactions, on doit identifier
        # les features de base d'origine pour régénérer les MÊMES interactions.
        # On régénère sur l'ensemble des features de base qui apparaissent,
        # seules ou en composant les noms d'interaction.
        base_feats_referenced = set(base_features_for_interactions)
        for f in ranked_features:
            if "_div_" in f:
                a, b = f.split("_div_")
                base_feats_referenced.update([a, b])
            elif "_minus_" in f:
                a, b = f.split("_minus_")
                base_feats_referenced.update([a, b])
        base_feats_referenced = [f for f in base_feats_referenced if f in df_full.columns]

        if base_feats_referenced:
            interactions_df_full = generate_pairwise_interactions(df_full, base_feats_referenced)
            df_full_with_interactions = pd.concat([df_full, interactions_df_full], axis=1)
            df_full_with_interactions = df_full_with_interactions.loc[:, ~df_full_with_interactions.columns.duplicated()]
        else:
            df_full_with_interactions = df_full

        missing_feats = [f for f in ranked_features if f not in df_full_with_interactions.columns]
        if missing_feats:
            print(f"[WARN] h={h} {regime}: {len(missing_feats)} features du cache absentes après "
                  f"régénération ({missing_feats[:5]}...). Elles seront ignorées.")
            ranked_features = [f for f in ranked_features if f in df_full_with_interactions.columns]

        if not ranked_features:
            print(f"[WARN] h={h} {regime}: plus aucune feature valide après régénération. Skip.")
            continue

        # --- Boucle walk-forward (RECOMMANDATION 1) ---
        for fold_idx, fold in enumerate(WALKFORWARD_FOLDS, 1):
            fold_train_end = pd.Timestamp(fold["train_end"])
            fold_test_start = pd.Timestamp(fold["test_start"])
            fold_test_end = pd.Timestamp(fold["test_end"])

            df_train_fold = df_full_with_interactions.loc[df_full_with_interactions.index < fold_train_end].copy()
            df_test_fold = df_full_with_interactions.loc[
                (df_full_with_interactions.index >= fold_test_start) &
                (df_full_with_interactions.index < fold_test_end)
            ].copy()

            if len(df_train_fold) < 30 or len(df_test_fold) < 5:
                print(f"[WARN] h={h} {regime} fold{fold_idx}: train ou test trop petit "
                      f"(train={len(df_train_fold)}, test={len(df_test_fold)}). Skip fold.")
                continue

            cleaner = TrainFittedCleaner()
            X_train_clean = cleaner.fit_transform(df_train_fold[ranked_features])
            X_test_clean = cleaner.transform(df_test_fold[ranked_features])

            scaler = StandardScaler()
            X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_clean), columns=ranked_features, index=df_train_fold.index)
            X_test_scaled = pd.DataFrame(scaler.transform(X_test_clean), columns=ranked_features, index=df_test_fold.index)
            y_train_series = pd.Series(df_train_fold["VIX_Direction"].values, index=df_train_fold.index)
            y_test_series = pd.Series(df_test_fold["VIX_Direction"].values, index=df_test_fold.index)

            if regime == "GLOBAL":
                train_mask_regime = pd.Series(True, index=df_train_fold.index)
                test_mask_regime = pd.Series(True, index=df_test_fold.index)
            else:
                train_mask_regime = df_train_fold["VIX_Regime"] == regime
                test_mask_regime = df_test_fold["VIX_Regime"] == regime

            y_train_regime_raw_full = y_train_series.loc[train_mask_regime].values
            X_test_regime_full = X_test_scaled.loc[test_mask_regime]
            y_test_regime = y_test_series.loc[test_mask_regime].values

            if len(y_train_regime_raw_full) < 30 or len(y_test_regime) < 5 or len(np.unique(y_train_regime_raw_full)) < 2:
                print(f"[WARN] h={h} {regime} fold{fold_idx}: échantillon régime trop petit "
                      f"(train={len(y_train_regime_raw_full)}, test={len(y_test_regime)}). Skip fold.")
                continue

            max_n = min(MAX_FEATURES_TO_TEST, len(ranked_features))
            min_n = min(MIN_N_FEATURES_TO_TEST, max_n)
            configs = model_configs()

            for algo_name in ALL_ALGOS:
                Model_class, param_grid, fixed_args = configs[algo_name]

                for n in range(min_n, max_n + 1):
                    feats_n = ranked_features[:n]

                    X_train_regime = X_train_scaled.loc[train_mask_regime, feats_n].copy()
                    y_train_regime_raw = y_train_regime_raw_full.copy()
                    X_test_regime = X_test_regime_full[feats_n].copy()

                    sm = SMOTE(random_state=RANDOM_STATE)
                    unique_classes, counts = np.unique(y_train_regime_raw, return_counts=True)
                    if len(unique_classes) > 1 and min(counts) > 1:
                        X_train_resampled, y_train_regime = sm.fit_resample(X_train_regime, y_train_regime_raw)
                        X_train_regime = pd.DataFrame(X_train_resampled, columns=feats_n)
                    else:
                        y_train_regime = y_train_regime_raw

                    cv = TimeSeriesSplit(n_splits=3)
                    try:
                        grid = GridSearchCV(Model_class(**fixed_args), param_grid=param_grid,
                                             scoring="roc_auc", cv=cv, n_jobs=-1)
                        grid.fit(X_train_regime.values, y_train_regime)
                        model = grid.best_estimator_
                    except Exception as e:
                        continue

                    pred = model.predict(X_test_regime.values)
                    proba = (model.predict_proba(X_test_regime.values)[:, 1]
                             if hasattr(model, "predict_proba") else pred.astype(float))
                    metrics = compute_metrics(y_test_regime, pred, proba)

                    model_number += 1
                    row = {
                        "Model_Number": model_number,
                        "Horizon_Days": h,
                        "Model": algo_name,
                        "VIX_Regime": regime,
                        "Train_Start_Used": train_start,
                        "Fold": fold_idx,
                        "Fold_Train_End": fold["train_end"],
                        "Fold_Test_Start": fold["test_start"],
                        "Fold_Test_End": fold["test_end"],
                        "Features": json.dumps(feats_n),
                        "N_Features": n,
                        "SMOTE_Used": True,
                    }
                    row.update(metrics)
                    final_rows.append(row)

            print(f"  [h={h} {regime} fold{fold_idx}] train_n={len(y_train_regime_raw_full)} "
                  f"test_n={len(y_test_regime)} - {len(ALL_ALGOS)} algos x N={min_n}..{max_n} testés.")

final_results_df = pd.DataFrame(final_rows)
print(f"\n[PHASE 2 DONE] {len(final_results_df)} lignes générées au total "
      f"({len(HORIZONS_TO_TEST)} horizons x {len(WALKFORWARD_FOLDS)} folds x régimes x algos x N).")


PHASE 2 (walk-forward) : entraînement final

--- h=1j CALM: fenêtre gagnante Train_Start=2001-01-01, 20 features ---
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 14.93, NORMAL [14.93-21.22), STRESS >= 21.22  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 253/6632
[TARGET] Remaining non-flat rows: 6379
VIX_Regime
NORMAL    2327
CALM      2045
STRESS    2007
Name: count, dtype: int64
[INTERACTIONS] Génération de paires pour 15 features de base (105 paires x 2 opérations = 210 colonnes max)...
[INTERACTIONS] 210 colonnes d'interactions générées.
[WARN] h=1 CALM: 14 features du cache absentes après régénération (['spx_drawdown_252d_div_DOW_Price_zscore_60d', 'spx_drawdown_252d_div_NASDAQ_Price_zscore_60d', 'vix_level_div_DOW_Price_zscore_60d', 'spx_drawdown_252d_div_XLK_Tech_zscore_60d', 'IVV_SP500_zscore_60d_div_NASDAQ_Price_vol_20d']...). Elles seront ignorées.
  [h=1 CALM fold1] train_n=1885 te

In [28]:
# =============================================================================
# Résumé : meilleure config (Modèle, N_Features) par (horizon, régime),
# évaluée sur la MOYENNE des 4 folds walk-forward (pas un seul split) -
# c'est la lecture qui répond directement à la Recommandation 1 : on veut
# une config stable across folds, pas juste performante sur un seul test set.
# =============================================================================

# Moyenne des métriques par (horizon, régime, modèle, N_Features) à travers les 4 folds
groupby_cols = ["Horizon_Days", "VIX_Regime", "Model", "N_Features"]
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

avg_across_folds_df = (
    final_results_df.groupby(groupby_cols)[metric_cols]
    .agg(["mean", "std"])
)
avg_across_folds_df.columns = ["_".join(c) for c in avg_across_folds_df.columns]
avg_across_folds_df = avg_across_folds_df.reset_index()

n_folds_present = final_results_df.groupby(groupby_cols).size().rename("N_Folds_Present")
avg_across_folds_df = avg_across_folds_df.merge(n_folds_present, on=groupby_cols)

avg_across_folds_df.to_csv(OUTPUT_DIR / "phase2_avg_across_folds.csv", index=False)
print(f"[SAVE] phase2_avg_across_folds.csv ({len(avg_across_folds_df)} lignes)")

# Meilleure config par (horizon, régime), classée sur AUC_mean (et pénalisée
# si peu de folds disponibles - on exige au moins 3 folds sur 4 pour être retenu)
best_per_horizon_regime_rows = []
for h in HORIZONS_TO_TEST:
    for regime in ALL_REGIMES:
        sub = avg_across_folds_df[
            (avg_across_folds_df["Horizon_Days"] == h) &
            (avg_across_folds_df["VIX_Regime"] == regime) &
            (avg_across_folds_df["N_Folds_Present"] >= 3)
        ]
        if sub.empty:
            print(f"[WARN] h={h} {regime}: aucune config avec >=3 folds valides. Skip.")
            continue
        best_row = sub.loc[sub["AUC_mean"].idxmax()]
        best_per_horizon_regime_rows.append(best_row)
        print(f"[BEST h={h}] {regime}: {best_row['Model']} N={best_row['N_Features']} "
              f"-> AUC={best_row['AUC_mean']:.4f} (+/-{best_row['AUC_std']:.4f}) "
              f"F1={best_row['F1_mean']:.4f} (+/-{best_row['F1_std']:.4f}) "
              f"sur {int(best_row['N_Folds_Present'])}/4 folds")

best_per_regime_df = pd.DataFrame(best_per_horizon_regime_rows)
display(best_per_regime_df[["Horizon_Days","VIX_Regime","Model","N_Features","N_Folds_Present",
                              "Accuracy_mean","Precision_mean","Recall_mean","F1_mean","F1_std",
                              "AUC_mean","AUC_std"]])


[SAVE] phase2_avg_across_folds.csv (390 lignes)
[BEST h=1] CALM: RandomForest N=5 -> AUC=0.6715 (+/-0.2056) F1=0.5952 (+/-0.0634) sur 3/4 folds
[BEST h=1] NORMAL: LightGBM N=5 -> AUC=0.6117 (+/-0.0318) F1=0.4815 (+/-0.0479) sur 3/4 folds
[BEST h=1] STRESS: RandomForest N=5 -> AUC=0.5523 (+/-0.1924) F1=0.3758 (+/-0.1411) sur 3/4 folds
[BEST h=1] GLOBAL: XGBoost N=5 -> AUC=0.6031 (+/-0.0257) F1=0.5050 (+/-0.0602) sur 3/4 folds
[WARN] h=3 CALM: aucune config avec >=3 folds valides. Skip.
[BEST h=3] NORMAL: LogisticRegression N=5 -> AUC=0.5614 (+/-0.0274) F1=0.6070 (+/-0.0813) sur 3/4 folds
[BEST h=3] STRESS: LightGBM N=6 -> AUC=0.5952 (+/-0.0925) F1=0.3286 (+/-0.1051) sur 3/4 folds
[BEST h=3] GLOBAL: GradientBoosting N=11 -> AUC=0.6445 (+/-0.0410) F1=0.5649 (+/-0.0466) sur 3/4 folds
[WARN] h=5 CALM: aucune config avec >=3 folds valides. Skip.
[BEST h=5] NORMAL: RandomForest N=8 -> AUC=0.5371 (+/-0.0328) F1=0.4382 (+/-0.1757) sur 3/4 folds
[BEST h=5] STRESS: LogisticRegression N=7 -> AUC=0

,Horizon_Days,VIX_Regime,Model,N_Features,N_Folds_Present,Accuracy_mean,Precision_mean,Recall_mean,F1_mean,F1_std,AUC_mean,AUC_std
6,1,CALM,RandomForest,5,3,0.582126,0.780303,0.495238,0.595217,0.063450,0.671469,0.205567
26,1,NORMAL,LightGBM,5,3,0.574489,0.512893,0.454493,0.481527,0.047945,0.611709,0.031818
72,1,STRESS,RandomForest,5,3,0.568708,0.342593,0.424242,0.375845,0.141119,0.552267,0.192395
22,1,GLOBAL,XGBoost,5,3,0.571338,0.520473,0.496733,0.504993,0.060220,0.603089,0.025736
166,3,NORMAL,LogisticRegression,5,3,0.564626,0.563609,0.670491,0.607016,0.081262,0.561426,0.027371
201,3,STRESS,LightGBM,6,3,0.595811,0.322807,0.340741,0.328642,0.105084,0.595202,0.092482
116,3,GLOBAL,GradientBoosting,11,3,0.602757,0.591516,0.551403,0.564857,0.046641,0.644539,0.040973
330,5,NORMAL,RandomForest,8,3,0.485850,0.520752,0.445466,0.438233,0.175693,0.537057,0.032833
359,5,STRESS,LogisticRegression,7,3,0.463448,0.177207,0.397306,0.244782,0.216626,0.499223,0.160974
300,5,GLOBAL,LogisticRegression,11,3,0.595891,0.583222,0.670566,0.616294,0.007803,0.676817,0.051658


In [29]:
# =============================================================================
# SUGGESTIONS AUTOMATIQUES D'AMELIORATION (multi-horizon, walk-forward)
# Compare chaque (horizon, régime) au meilleur modèle de référence connu du
# projet (horizon 1j, split fixe unique - les seules valeurs de référence
# disponibles), et ajoute des diagnostics spécifiques au walk-forward
# (stabilité inter-folds) et au choix d'horizon.
# =============================================================================

REFERENCE_METRICS = {
    "CALM":   {"Model": "RandomForest",      "F1": 0.667, "AUC": 0.615},
    "NORMAL": {"Model": "GradientBoosting",  "F1": 0.564, "AUC": 0.608},
    "STRESS": {"Model": "XGBoost (SMOTE)",   "F1": 0.470, "AUC": 0.613},
    "GLOBAL": {"Model": "RandomForest (SMOTE)", "F1": 0.545, "AUC": 0.604},
}

suggestions = []

for _, best_row in best_per_regime_df.iterrows():
    h = best_row["Horizon_Days"]
    regime = best_row["VIX_Regime"]
    ref = REFERENCE_METRICS.get(regime)
    if ref is None:
        continue

    f1_gap = best_row["F1_mean"] - ref["F1"]
    auc_gap = best_row["AUC_mean"] - ref["AUC"]

    if f1_gap > 0.01 and auc_gap > 0.005:
        verdict = "AMELIORATION CONFIRMEE"
    elif f1_gap < -0.02 or auc_gap < -0.01:
        verdict = "REGRESSION - garder le modele de reference existant"
    else:
        verdict = "ECART MARGINAL - pas de gain net clair"

    suggestions.append({
        "Horizon_Days": h,
        "VIX_Regime": regime,
        "Verdict": verdict,
        "Nouveau_Modele": f"{best_row['Model']} (N={best_row['N_Features']})",
        "Modele_Reference": f"{ref['Model']} (horizon 1j, split fixe)",
        "F1_Nouveau_Mean": round(best_row["F1_mean"], 4),
        "F1_Nouveau_Std": round(best_row["F1_std"], 4) if not pd.isna(best_row["F1_std"]) else None,
        "F1_Reference": ref["F1"],
        "F1_Gap": round(f1_gap, 4),
        "AUC_Nouveau_Mean": round(best_row["AUC_mean"], 4),
        "AUC_Nouveau_Std": round(best_row["AUC_std"], 4) if not pd.isna(best_row["AUC_std"]) else None,
        "AUC_Reference": ref["AUC"],
        "AUC_Gap": round(auc_gap, 4),
    })

    diagnostics = []

    # 1. Stabilité inter-folds (spécifique walk-forward) : AUC_std élevé =
    #    performance qui varie beaucoup d'un fold à l'autre, signe de fragilité.
    if not pd.isna(best_row["AUC_std"]) and best_row["AUC_std"] > 0.05:
        diagnostics.append(
            f"AUC_std={best_row['AUC_std']:.3f} eleve a travers les 4 folds walk-forward -> "
            f"performance instable dans le temps, ne pas se fier au seul AUC moyen ; "
            f"ce modele pourrait bien fonctionner sur 2024 et mal sur 2025 (ou l'inverse)."
        )
    if best_row["N_Folds_Present"] < 4:
        diagnostics.append(
            f"Seulement {int(best_row['N_Folds_Present'])}/4 folds disponibles pour cette config "
            f"-> echantillon insuffisant sur certains folds (probablement {regime} a peu "
            f"d'observations sur certaines periodes), resultat a interpreter avec prudence."
        )

    for d in diagnostics:
        suggestions.append({"Horizon_Days": h, "VIX_Regime": regime, "Verdict": "DIAGNOSTIC",
                             "Nouveau_Modele": d, "Modele_Reference": "", "F1_Nouveau_Mean": None,
                             "F1_Nouveau_Std": None, "F1_Reference": None, "F1_Gap": None,
                             "AUC_Nouveau_Mean": None, "AUC_Nouveau_Std": None,
                             "AUC_Reference": None, "AUC_Gap": None})

# --- Comparaison entre horizons, par régime : quel horizon est le meilleur ? ---
horizon_comparison_rows = []
for regime in ALL_REGIMES:
    sub = best_per_regime_df[best_per_regime_df["VIX_Regime"] == regime]
    if sub.empty:
        continue
    best_h_row = sub.loc[sub["AUC_mean"].idxmax()]
    horizon_comparison_rows.append({
        "VIX_Regime": regime,
        "Meilleur_Horizon_Days": int(best_h_row["Horizon_Days"]),
        "AUC_Mean": round(best_h_row["AUC_mean"], 4),
        "F1_Mean": round(best_h_row["F1_mean"], 4),
    })
horizon_comparison_df = pd.DataFrame(horizon_comparison_rows)

general_suggestions = [
    "Walk-forward : la stabilite inter-folds (AUC_std) est maintenant mesurable - privilegier "
    "systematiquement les configs a faible std plutot que le seul AUC moyen le plus eleve, "
    "surtout si l'usage final est un suivi macro continu plutot qu'un backtest ponctuel.",
    "Horizons multiples : si un horizon plus long (3j ou 5j) domine systematiquement sur "
    "plusieurs regimes, ca suggere que le signal a 1 jour est structurellement plus bruite "
    "et qu'un usage operationnel a horizon plus long serait plus robuste.",
    "Interactions (Recommandation 2) : verifier la colonne N_Interactions_Kept du fichier "
    "phase1_feature_quality_by_window_and_horizon.csv - si elle est proche de 0 partout, les "
    "interactions ratio/difference n'apportent pas de signal au-dela des features brutes pour "
    "ce projet, et le cout de calcul associe n'est pas justifie pour les prochains runs.",
    "Folds adaptatifs : verifier la duree effective de chaque fold (colonne Fold_Test_Start / "
    "Fold_Test_End du detail Phase2_All_Results) - si certains folds deviennent tres longs pour "
    "atteindre MIN_OBS_PER_REGIME_PER_FOLD, ca signale des periodes prolongees sans episode "
    "CALM ou STRESS, utile a documenter independamment du resultat des modeles.",
]

suggestions_df = pd.DataFrame(suggestions)
print("="*80)
print("SUGGESTIONS D'AMELIORATION (générées à partir de ce run)")
print("="*80)
for h in HORIZONS_TO_TEST:
    for regime in ALL_REGIMES:
        sub_sugg = suggestions_df[(suggestions_df["Horizon_Days"] == h) & (suggestions_df["VIX_Regime"] == regime)]
        if sub_sugg.empty:
            continue
        print(f"\n--- horizon={h}j régime={regime} ---")
        for _, row in sub_sugg.iterrows():
            if row["Verdict"] == "DIAGNOSTIC":
                print(f"  [DIAGNOSTIC] {row['Nouveau_Modele']}")
            else:
                print(f"  [{row['Verdict']}] {row['Nouveau_Modele']} vs {row['Modele_Reference']} -> "
                      f"F1 {row['F1_Nouveau_Mean']} vs ref {row['F1_Reference']} (gap {row['F1_Gap']:+.4f})")

print("\n--- Meilleur horizon par régime ---")
display(horizon_comparison_df)

print("\n--- Suggestions générales ---")
for s in general_suggestions:
    print(f"  - {s}")

general_suggestions_df = pd.DataFrame({"Suggestion_Generale": general_suggestions})


SUGGESTIONS D'AMELIORATION (générées à partir de ce run)

--- horizon=1j régime=CALM ---
  [REGRESSION - garder le modele de reference existant] RandomForest (N=5) vs RandomForest (horizon 1j, split fixe) -> F1 0.5952 vs ref 0.667 (gap -0.0718)
  [DIAGNOSTIC] AUC_std=0.206 eleve a travers les 4 folds walk-forward -> performance instable dans le temps, ne pas se fier au seul AUC moyen ; ce modele pourrait bien fonctionner sur 2024 et mal sur 2025 (ou l'inverse).
  [DIAGNOSTIC] Seulement 3/4 folds disponibles pour cette config -> echantillon insuffisant sur certains folds (probablement CALM a peu d'observations sur certaines periodes), resultat a interpreter avec prudence.

--- horizon=1j régime=NORMAL ---
  [REGRESSION - garder le modele de reference existant] LightGBM (N=5) vs GradientBoosting (horizon 1j, split fixe) -> F1 0.4815 vs ref 0.564 (gap -0.0825)
  [DIAGNOSTIC] Seulement 3/4 folds disponibles pour cette config -> echantillon insuffisant sur certains folds (probablement NORMA

,VIX_Regime,Meilleur_Horizon_Days,AUC_Mean,F1_Mean
0,CALM,1,0.6715,0.5952
1,NORMAL,1,0.6117,0.4815
2,STRESS,3,0.5952,0.3286
3,GLOBAL,5,0.6768,0.6163



--- Suggestions générales ---
  - Walk-forward : la stabilite inter-folds (AUC_std) est maintenant mesurable - privilegier systematiquement les configs a faible std plutot que le seul AUC moyen le plus eleve, surtout si l'usage final est un suivi macro continu plutot qu'un backtest ponctuel.
  - Horizons multiples : si un horizon plus long (3j ou 5j) domine systematiquement sur plusieurs regimes, ca suggere que le signal a 1 jour est structurellement plus bruite et qu'un usage operationnel a horizon plus long serait plus robuste.
  - Interactions (Recommandation 2) : verifier la colonne N_Interactions_Kept du fichier phase1_feature_quality_by_window_and_horizon.csv - si elle est proche de 0 partout, les interactions ratio/difference n'apportent pas de signal au-dela des features brutes pour ce projet, et le cout de calcul associe n'est pas justifie pour les prochains runs.
  - Folds adaptatifs : verifier la duree effective de chaque fold (colonne Fold_Test_Start / Fold_Test_End du det

In [30]:
# =============================================================================
# UN SEUL fichier Excel final, avec 8 feuilles :
#   - Phase1_Feature_Scores  : score composite par (horizon, train_start, régime)
#   - Phase1_Winning_Windows : fenêtre gagnante par (horizon, régime)
#   - Phase2_All_Results     : tous les modèles entraînés (horizons x folds x régimes x algos x N)
#   - Phase2_Avg_Across_Folds: moyenne/std des métriques par config, à travers les 4 folds
#   - Best_Model_Per_Regime  : meilleure config par (horizon, régime), avec stabilité inter-folds
#   - Horizon_Comparison     : meilleur horizon par régime
#   - Suggestions_Par_Regime : verdict + diagnostics générés automatiquement
#   - Suggestions_Generales  : suggestions transverses
# =============================================================================

excel_filename = OUTPUT_DIR / "vix_walkforward_multihorizon_report.xlsx"

with pd.ExcelWriter(excel_filename, engine="xlsxwriter") as writer:
    feature_quality_df.to_excel(writer, sheet_name="Phase1_Feature_Scores", index=False)
    winner_summary_df.to_excel(writer, sheet_name="Phase1_Winning_Windows", index=False)
    final_results_df.to_excel(writer, sheet_name="Phase2_All_Results", index=False)
    avg_across_folds_df.to_excel(writer, sheet_name="Phase2_Avg_Across_Folds", index=False)
    best_per_regime_df.to_excel(writer, sheet_name="Best_Model_Per_Regime", index=False)
    horizon_comparison_df.to_excel(writer, sheet_name="Horizon_Comparison", index=False)
    suggestions_df.to_excel(writer, sheet_name="Suggestions_Par_Regime", index=False)
    general_suggestions_df.to_excel(writer, sheet_name="Suggestions_Generales", index=False)

print(f"[SAVE] Fichier Excel unique -> {excel_filename}")
print(f"[RECAP]")
print(f"  - Phase1_Feature_Scores   : {len(feature_quality_df)} lignes")
print(f"  - Phase1_Winning_Windows  : {len(winner_summary_df)} lignes")
print(f"  - Phase2_All_Results      : {len(final_results_df)} lignes")
print(f"  - Phase2_Avg_Across_Folds : {len(avg_across_folds_df)} lignes")
print(f"  - Best_Model_Per_Regime   : {len(best_per_regime_df)} lignes")
print(f"  - Horizon_Comparison      : {len(horizon_comparison_df)} lignes")
print(f"  - Suggestions_Par_Regime  : {len(suggestions_df)} lignes")
print(f"  - Suggestions_Generales   : {len(general_suggestions_df)} lignes")


[SAVE] Fichier Excel unique -> /content/outputs_v20_balanced_folds_no_trends/vix_walkforward_multihorizon_report.xlsx
[RECAP]
  - Phase1_Feature_Scores   : 132 lignes
  - Phase1_Winning_Windows  : 12 lignes
  - Phase2_All_Results      : 1120 lignes
  - Phase2_Avg_Across_Folds : 390 lignes
  - Best_Model_Per_Regime   : 10 lignes
  - Horizon_Comparison      : 4 lignes
  - Suggestions_Par_Regime  : 25 lignes
  - Suggestions_Generales   : 4 lignes
